# **SmolDocling : Powerful OCR**

In [1]:
# Install all the required packages
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!pip install einops timm pillow
!pip install git+https://github.com/huggingface/transformers
!pip install git+https://github.com/huggingface/accelerate
!pip install git+https://github.com/huggingface/diffusers
!pip install huggingface_hub sentencepiece bitsandbytes protobuf decord numpy
!pip install docling_core # Assuming this is the correct package name

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 48.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjit

In [3]:
import torch
print(torch.version.cuda)   # should not be None
print(torch.cuda.is_available())

12.4
True


In [4]:
pip install flash-attn --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 62.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp311-cp311-linux_x86_64.whl size=256022485 sha256=0abc62d04f28f140f4f76ab7cfd1d8ce24a69c6ab0cbace8d4ab99640b68dc0a
  Stored in directory: /root/.cache/pip/wheels/42/31/1f/4b22dd7295b3cb064b8fa9038f6d58fb15c9571555b2d7c39c
Successfully built flash-attn


In [6]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers.image_utils import load_image
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained("ds4sd/SmolDocling-256M-preview")

# Use attn_implementation instead of _attn_implementation
# Fallback to "eager" if flash-attn fails
try:
    model = AutoModelForVision2Seq.from_pretrained(
        "ds4sd/SmolDocling-256M-preview",
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map="auto"
    )
except Exception as e:
    print("⚠️ FlashAttention2 failed, falling back to eager mode:", e)
    model = AutoModelForVision2Seq.from_pretrained(
        "ds4sd/SmolDocling-256M-preview",
        torch_dtype=torch.bfloat16,
        attn_implementation="eager",
        device_map="auto"
    )

⚠️ FlashAttention2 failed, falling back to eager mode: /usr/local/lib/python3.11/dist-packages/flash_attn_2_cuda.cpython-311-x86_64-linux-gnu.so: undefined symbol: _ZN3c105ErrorC2ENS_14SourceLocationENSt7__cxx1112basic_stringIcSt11char_traitsIcESaIcEEE


generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

## **Paragraph Raeading**

In [13]:
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument

image = Image.open('/content/eng_paragraph.png').convert('RGB')

messages = [{"role": "user",
             "content": [{"type": "image"}, {"type": "text", "text": "Convert this page to docling."}]}]

prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

generated_ids = model.generate(**inputs, max_new_tokens=8192)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doctags = processor.batch_decode(trimmed_generated_ids, skip_special_tokens=False)[0].lstrip()

doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
print(doctags)

doc = DoclingDocument(name="Document")
doc.load_from_doctags(doctags_doc)
# print(doc.export_to_markdown())

from IPython.display import display, Markdown
display(Markdown(doc.export_to_markdown()))

<doctag><text><loc_0><loc_1><loc_473><loc_100>Configure your code by storing environment variables, file paths, or keys. Values stored here are private, visible only to you and the notebooks that you select.</text>
<text><loc_0><loc_124><loc_265><loc_150>Secret name cannot contain spaces.</text>
</doctag><end_of_utterance>


## **Math Equation reading**

In [15]:
image = Image.open('/content/math_eq.png').convert('RGB')

messages = [{"role": "user",
             "content": [{"type": "image"}, {"type": "text", "text": "Convert this page to docling."}]}]

prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

generated_ids = model.generate(**inputs, max_new_tokens=8192)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doctags = processor.batch_decode(trimmed_generated_ids, skip_special_tokens=False)[0].lstrip()

doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
print(doctags)

doc = DoclingDocument(name="Document")
doc.load_from_doctags(doctags_doc)

#print(doc.export_to_markdown())
#print(doc.export_to_html())
from IPython.display import Markdown, display
display(Markdown(doc.export_to_markdown()))

<doctag><text><loc_54><loc_130><loc_385><loc_238>test() = 5 + 2 . 3 - \frac { 2 0 } { 5 }</text>
</doctag><end_of_utterance>


# **Table Reading**

In [16]:


image = Image.open('/content/table.png').convert('RGB')

messages = [{"role": "user",
             "content": [{"type": "image"}, {"type": "text", "text": "Convert this page to docling."}]}]

prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

generated_ids = model.generate(**inputs, max_new_tokens=8192)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doctags = processor.batch_decode(trimmed_generated_ids, skip_special_tokens=False)[0].lstrip()

doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
print(doctags)

doc = DoclingDocument(name="Document")
doc.load_from_doctags(doctags_doc)

#print(doc.export_to_markdown())
#print(doc.export_to_html())
from IPython.display import Markdown, display
display(Markdown(doc.export_to_markdown()))


<doctag><otsl><loc_0><loc_0><loc_500><loc_500><ched>Id<ched>Name<ched>Email<ched>Investments<nl><fcel>231<fcel>Albert Master<fcel>albert.master@gmail.com<fcel>Bonds<nl><fcel>210<fcel>Alfred Alan<fcel>aalan@gmail.com<fcel>Stocks<nl><fcel>256<fcel>Alison Smart<fcel>asmart@biztalk.com<fcel>Residential Property<nl><fcel>211<fcel>Ally Emery<fcel>ally@easymail.com<fcel>Stocks<nl><fcel>248<fcel>Andrew Phibis<fcel>andy@mycorp.com<fcel>Stocks<nl><fcel>234<fcel>Andy Mitchell<fcel>andy@hotmail.com<fcel>Stocks<nl><fcel>226<fcel>Angus Robins<fcel>arobins@robins.com<fcel>Bonds<nl><fcel>241<fcel>Ann Melan<fcel>ann_melan@inet.com<fcel>Residential Property<nl><fcel>225<fcel>Ben Bessel<fcel>benb@hotmail.com<fcel>Stocks<nl><fcel>235<fcel>Ben Bensen Romanolf<fcel>benr@albert.net<fcel>Bonds<nl></otsl><end_of_utterance>


## **Chart Reading**

In [18]:
image = Image.open('/content/chart.png').convert('RGB')

messages = [{"role": "user",
             "content": [{"type": "image"}, {"type": "text", "text": "Convert chart to table."}]}]

prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

generated_ids = model.generate(**inputs, max_new_tokens=8192)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doctags = processor.batch_decode(trimmed_generated_ids, skip_special_tokens=False)[0].lstrip()

doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
print(doctags)

doc = DoclingDocument(name="Document")
doc.load_from_doctags(doctags_doc)

#print(doc.export_to_markdown())
#print(doc.export_to_html())
from IPython.display import Markdown, display
display(Markdown(doc.export_to_markdown()))

<chart><loc_0><loc_0><loc_500><loc_500><unknown><fcel>Characteristic<fcel>Value<nl><fcel>Unsafe sex<fcel>1.31 million<nl><fcel>Alcohol use<fcel>880,756<nl><fcel>High blood pressure<fcel>619,328<nl><fcel>Smoking<fcel>597,653<nl><fcel>High body-mass index (obesity)<fcel>1099,812<nl><fcel>High blood sugar<fcel>334,864<nl><fcel>Diet low in fruits<fcel>316,783<nl><fcel>Drug use<fcel>226,833<nl><fcel>Diet low in vegetables<fcel>200,275<nl><fcel>Outdoor air pollution<fcel>194,601<nl><fcel>Household air pollution<fcel>181,256<nl><fcel>Unsafe water source<fcel>122,777<nl><fcel>Secondhand smoke<fcel>105,318<nl><fcel>Iron deficiency<fcel>96,915<nl><fcel>Poor sanitation<fcel>92,072<nl><fcel>No access to handwashing facility<fcel>68,467<nl><fcel>Low physical activity<fcel>48,930<nl><fcel>Low bone mineral density<fcel>13,135<nl><fcel>Source: IHME, Global Burden of Disease (GBD)<fcel>1.2 million<nl></otsl><end_of_utterance>


## **Code reading**

In [19]:
image = Image.open('/content/code.png').convert('RGB')

messages = [{"role": "user",
             "content": [{"type": "image"}, {"type": "text", "text": "Convert code to text.."}]}]

prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

generated_ids = model.generate(**inputs, max_new_tokens=8192)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doctags = processor.batch_decode(trimmed_generated_ids, skip_special_tokens=False)[0].lstrip()

doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
print(doctags)

doc = DoclingDocument(name="Document")
doc.load_from_doctags(doctags_doc)

#print(doc.export_to_markdown())
#print(doc.export_to_html())
from IPython.display import Markdown, display
display(Markdown(doc.export_to_markdown()))

<code><loc_0><loc_0><loc_500><loc_500><_Python_> image = Image.open('/content/code.png').convert('RGB')

message = [r"role": "user",
    "content": [{"type": "image"}, {"type": "text", "text": "Convert this page to docling."}]]

prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor.text=prompt,images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

generated_ids = model.generate(**inputs, max_new_tokens=8192)
prompt_length = generated_ids.input_ids.shape[1:]
trimmed_generated_ids = generated_ids[:1]
doctags = processor.batch_decode(trimmed_generated_ids, skip_special_tokens=False)[0].lstrip()

doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
print(doctags)


doc = DoclingDocument(name="Document")
doc.load_from_doctags(doctags_doc)

#print(doc.export_to_markdown())
#print(doc.export_to_html())
from IPython.display import Markdown, display
display(Markdown(doc.export_to_markdown()))
</code><end_of_utterance>

### **Limitations**

*   no multi-lang support
*   Developed by: Docling Team, IBM Research
*   Model type: Multi-modal model (image+text)
*   Language(s) (NLP): English






# **using Simple Docling instead of SmolDocling**

In [20]:
pip install docling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.6/86.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 43.7 

In [21]:
from docling.document_converter import DocumentConverter

source = "https://arxiv.org/pdf/2408.09869"  # document per local path or URL
converter = DocumentConverter()
result = converter.convert(source)
print(result.document.export_to_markdown())  # output: "## Docling Technical Report[...]"

<!-- image -->

## Docling Technical Report

Version 1.0

Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta Rafael Teixeira de Lima Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. J. Staar

AI4K Group, IBM Research R¨ uschlikon, Switzerland

## Abstract

This technical report introduces Docling , an easy to use, self-contained, MITlicensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. The code interface allows for easy extensibility and addition of new features and models.

## 1 Introduction

Converting PDF documents back into a machine-processable format has been a major challenge for decades due to their huge variabi